In [6]:
from pathlib import Path
import csv
import torch

from ultralytics import YOLO


# ============================================================
# Configuration
# ============================================================

DATA_YAML = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\data.yaml"
)

PROJECT_DIR = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation"
    r"\YOLO_Experiments"
)

EXPERIMENT_NAME = "YOLOv9m_Axial_Localization"

# Your images are originally 256 x 256.
IMAGE_SIZE = 256

EPOCHS = 100
BATCH_SIZE = 16
WORKERS = 4
DEVICE = 0

PATIENCE = 40
RANDOM_SEED = 42


# ============================================================
# Utility functions
# ============================================================

def print_environment_information() -> None:
    """
    Print PyTorch and GPU information.
    """
    print("=" * 70)
    print("ENVIRONMENT INFORMATION")
    print("=" * 70)

    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU count: {torch.cuda.device_count()}")
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
    else:
        print(
            "[WARNING] CUDA is not available. "
            "Training will run on the CPU."
        )


def validate_paths() -> None:
    """
    Check that the dataset configuration exists.
    """
    if not DATA_YAML.exists():
        raise FileNotFoundError(
            f"data.yaml was not found:\n{DATA_YAML}"
        )

    PROJECT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


def save_metrics_to_csv(
    metrics,
    output_csv: Path,
    split_name: str
) -> None:
    """
    Save object-detection metrics returned by model.val().
    """
    output_csv.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    results_dictionary = getattr(
        metrics,
        "results_dict",
        {}
    )

    row = {
        "split": split_name,
        "precision": results_dictionary.get(
            "metrics/precision(B)",
            ""
        ),
        "recall": results_dictionary.get(
            "metrics/recall(B)",
            ""
        ),
        "mAP50": results_dictionary.get(
            "metrics/mAP50(B)",
            ""
        ),
        "mAP50_95": results_dictionary.get(
            "metrics/mAP50-95(B)",
            ""
        ),
        "fitness": results_dictionary.get(
            "fitness",
            ""
        ),
    }

    with output_csv.open(
        mode="w",
        encoding="utf-8",
        newline=""
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=list(row.keys())
        )

        writer.writeheader()
        writer.writerow(row)


def print_detection_metrics(
    metrics,
    split_name: str
) -> None:
    """
    Print overall and per-class detection results.
    """
    print("\n" + "=" * 70)
    print(f"{split_name.upper()} RESULTS")
    print("=" * 70)

    box_metrics = metrics.box

    print(f"Precision:  {box_metrics.mp:.6f}")
    print(f"Recall:     {box_metrics.mr:.6f}")
    print(f"mAP@0.5:    {box_metrics.map50:.6f}")
    print(f"mAP@0.75:   {box_metrics.map75:.6f}")
    print(f"mAP@0.5:0.95: {box_metrics.map:.6f}")

    class_names = metrics.names

    print("\nPer-class mAP@0.5:0.95:")

    for class_id, class_map in enumerate(box_metrics.maps):
        class_name = class_names.get(
            class_id,
            str(class_id)
        )

        print(
            f"  {class_id} - {class_name}: "
            f"{float(class_map):.6f}"
        )


# ============================================================
# Training
# ============================================================

def train_model() -> Path:
    """
    Train YOLOv9m and return the best checkpoint path.
    """
    print("\n" + "=" * 70)
    print("TRAINING YOLOv9m")
    print("=" * 70)

    # Ultralytics YOLOv9 medium pretrained detector.
    model = YOLO("yolov9s.pt")

    model.train(
        data=str(DATA_YAML),

        # Training duration
        epochs=EPOCHS,
        patience=PATIENCE,

        # Input and hardware
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,

        # Output
        project=str(PROJECT_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=True,

        # Reproducibility
        seed=RANDOM_SEED,
        deterministic=True,

        # Optimization
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,

        # Dataset settings
        cache=False,
        rect=False,
        single_cls=False,

        # Augmentations suitable for axial MRI
        degrees=5.0,
        translate=0.05,
        scale=0.10,
        shear=0.0,
        perspective=0.0,

        # Horizontal flipping is disabled because left and right
        # are separate classes: LLS/RLS and LFS/RFS.
        fliplr=0.0,
        flipud=0.0,

        # Color augmentation is disabled for grayscale MRI.
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.05,

        # Strong mixing augmentations can be harmful for anatomical MRI.
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,

        # Save settings
        save=True,
        save_period=-1,
        plots=True,
        verbose=True,
    )

    best_model_path = (
        PROJECT_DIR
        / EXPERIMENT_NAME
        / "weights"
        / "best.pt"
    )

    if not best_model_path.exists():
        raise FileNotFoundError(
            "Training finished, but best.pt was not found:\n"
            f"{best_model_path}"
        )

    print(f"\nBest model:\n{best_model_path}")

    return best_model_path


# ============================================================
# Validation and testing
# ============================================================

def evaluate_model(
    best_model_path: Path,
    split_name: str
):
    """
    Evaluate best.pt on the validation or test split.
    """
    if split_name not in {"val", "test"}:
        raise ValueError(
            "split_name must be either 'val' or 'test'."
        )

    print("\n" + "=" * 70)
    print(f"EVALUATING {split_name.upper()} SPLIT")
    print("=" * 70)

    model = YOLO(str(best_model_path))

    metrics = model.val(
        data=str(DATA_YAML),
        split=split_name,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,

        # Detection evaluation settings
        conf=0.001,
        iou=0.6,
        max_det=20,

        # Save evaluation figures
        plots=True,
        save_json=False,

        project=str(PROJECT_DIR),
        name=f"{EXPERIMENT_NAME}_{split_name}_evaluation",
        exist_ok=True,
        verbose=True,
    )

    print_detection_metrics(
        metrics=metrics,
        split_name=split_name
    )

    output_csv = (
        PROJECT_DIR
        / EXPERIMENT_NAME
        / f"{split_name}_metrics.csv"
    )

    save_metrics_to_csv(
        metrics=metrics,
        output_csv=output_csv,
        split_name=split_name
    )

    print(f"\nMetrics saved to:\n{output_csv}")

    return metrics


# ============================================================
# Test-set prediction visualization
# ============================================================

def predict_test_images(
    best_model_path: Path
) -> None:
    """
    Run inference on all test images and save visual predictions.
    """
    test_images_directory = (
        DATA_YAML.parent
        / "images"
        / "test"
    )

    if not test_images_directory.exists():
        raise FileNotFoundError(
            f"Test images directory was not found:\n"
            f"{test_images_directory}"
        )

    print("\n" + "=" * 70)
    print("GENERATING TEST PREDICTIONS")
    print("=" * 70)

    model = YOLO(str(best_model_path))

    model.predict(
        source=str(test_images_directory),
        imgsz=IMAGE_SIZE,
        conf=0.25,
        iou=0.6,
        max_det=20,
        device=DEVICE,

        # Save visual predictions.
        save=True,

        # Save YOLO prediction TXT files.
        save_txt=True,

        # Save confidence scores in TXT files.
        save_conf=True,

        # Do not crop detections.
        save_crop=False,

        # Display labels and confidence.
        show_labels=True,
        show_conf=True,
        show_boxes=True,

        project=str(PROJECT_DIR),
        name=f"{EXPERIMENT_NAME}_test_predictions",
        exist_ok=True,
        verbose=True,
    )

    prediction_directory = (
        PROJECT_DIR
        / f"{EXPERIMENT_NAME}_test_predictions"
    )

    print(
        f"\nTest predictions saved to:\n"
        f"{prediction_directory}"
    )


# ============================================================
# Main
# ============================================================

def main() -> None:
    print_environment_information()
    validate_paths()

    best_model_path = train_model()

    # Evaluate on validation set.
    evaluate_model(
        best_model_path=best_model_path,
        split_name="val"
    )

    # Final independent evaluation on the test set.
    evaluate_model(
        best_model_path=best_model_path,
        split_name="test"
    )

    # Save bounding-box visualizations for all test images.
    predict_test_images(
        best_model_path=best_model_path
    )

    print("\n" + "=" * 70)
    print("TRAINING AND TESTING COMPLETED")
    print("=" * 70)

    print(f"\nBest model:\n{best_model_path}")

    print(
        "\nTraining results:\n"
        f"{PROJECT_DIR / EXPERIMENT_NAME}"
    )

    print(
        "\nTest evaluation:\n"
        f"{PROJECT_DIR / f'{EXPERIMENT_NAME}_test_evaluation'}"
    )

    print(
        "\nTest predictions:\n"
        f"{PROJECT_DIR / f'{EXPERIMENT_NAME}_test_predictions'}"
    )


if __name__ == "__main__":
    main()

ENVIRONMENT INFORMATION
PyTorch version: 2.11.0.dev20260128+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 5070 Ti

TRAINING YOLOv9m
New https://pypi.org/project/ultralytics/8.4.95 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.179  Python-3.12.7 torch-2.11.0.dev20260128+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.

train: Scanning D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\train... 1745 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1745/1745 [00:01<00:00, 1418.47it/s]

train: New cache created: D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access  (ping: 0.00.0 ms, read: 13.513.2 MB/s, size: 69.4 KB)


val: Scanning D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\val... 247 images, 0 backgrounds, 0 corrupt: 100%|██████████| 247/247 [00:00<00:00, 1518.52it/s]

...

                   all        247       1235      0.966      0.967      0.964      0.521
                   CCS        247        247          1      0.995      0.995      0.621
                   LLS        247        247      0.982      0.984       0.98      0.535
                   RLS        247        247      0.983      0.984      0.986      0.555
                   RFS        247        247      0.926      0.935      0.917      0.424
                   LFS        247        247      0.939      0.939      0.941      0.468
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Experiments\YOLOv9m_Axial_Localization_val_evaluation

VAL RESULTS
Precision:  0.966008
Recall:     0.967374
mAP@0.5:    0.963663
mAP@0.75:   0.495777
mAP@0.5:0.95: 0.520594

Per-class mAP@0.5:0.95:
  0 - CCS: 0.620784
  1 - LLS: 0.534650
  2 - RLS: 0.554922
  3 - RFS: 0.424

val: Scanning D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\test... 499 images, 0 backgrounds, 0 corrupt: 100%|██████████| 499/499 [00:00<00:00, 1803.26it/s]

val: New cache created: D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [00:02<00:00, 15.68it/s]


                   all        499       2491      0.971      0.969      0.976      0.529
                   CCS        499        499      0.998      0.987      0.995       0.62
                   LLS        498        498       0.99      0.983      0.993      0.556
                   RLS        498        498       0.97      0.969      0.979      0.546
                   RFS        498        498      0.927      0.934      0.937      0.454
                   LFS        498        498      0.968      0.972      0.975      0.467
Speed: 0.0ms preprocess, 1.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Experiments\YOLOv9m_Axial_Localization_test_evaluation

TEST RESULTS
Precision:  0.970541
Recall:     0.968844
mAP@0.5:    0.975775
mAP@0.75:   0.502219
mAP@0.5:0.95: 0.528553

Per-class mAP@0.5:0.95:
  0 - CCS: 0.619989
  1 - LLS: 0.556231
  2 - RLS: 0.545871
  3 - RFS: 0.4

###  Testing 

In [7]:
from pathlib import Path
import csv

import cv2
import numpy as np
from ultralytics import YOLO


# ============================================================
# Configuration
# ============================================================

DATA_YAML = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code"
    r"\technical validation\YOLO_Axial_Localization\data.yaml"
)

PROJECT_DIR = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code"
    r"\technical validation\YOLO_Experiments"
)

EXPERIMENT_NAME = "YOLOv9m_Axial_Localization"

BEST_CHECKPOINT = (
    PROJECT_DIR
    / EXPERIMENT_NAME
    / "weights"
    / "best.pt"
)

TEST_IMAGES = (
    DATA_YAML.parent
    / "images"
    / "test"
)

TEST_LABELS = (
    DATA_YAML.parent
    / "labels"
    / "test"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / f"{EXPERIMENT_NAME}_test_evaluation"
)

IMAGE_SIZE = 256
BATCH_SIZE = 16
WORKERS = 4
DEVICE = 0

# Prediction confidence threshold
CONF_THRESHOLD = 0.25

# Prediction is correct when IoU >= 0.45
MATCH_IOU_THRESHOLD = 0.45

# NMS IoU threshold
NMS_IOU_THRESHOLD = 0.60

MAX_DETECTIONS = 20


# ============================================================
# IoU calculation
# ============================================================

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes.

    Box format:
        [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0.0, x2 - x1)
    intersection_height = max(0.0, y2 - y1)

    intersection_area = intersection_width * intersection_height

    box1_area = max(0.0, box1[2] - box1[0]) * max(
        0.0,
        box1[3] - box1[1]
    )

    box2_area = max(0.0, box2[2] - box2[0]) * max(
        0.0,
        box2[3] - box2[1]
    )

    union_area = box1_area + box2_area - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


# ============================================================
# Load YOLO ground-truth labels
# ============================================================

def load_ground_truth(label_path, image_width, image_height):
    """
    Load YOLO-format labels and convert normalized xywh boxes
    into pixel xyxy boxes.

    Returns:
        [
            {
                "class_id": int,
                "box": [x1, y1, x2, y2]
            }
        ]
    """

    ground_truth = []

    if not label_path.exists():
        return ground_truth

    with label_path.open("r", encoding="utf-8") as file:
        for line in file:
            values = line.strip().split()

            if len(values) < 5:
                continue

            class_id = int(float(values[0]))

            x_center = float(values[1]) * image_width
            y_center = float(values[2]) * image_height
            box_width = float(values[3]) * image_width
            box_height = float(values[4]) * image_height

            x1 = x_center - box_width / 2
            y1 = y_center - box_height / 2
            x2 = x_center + box_width / 2
            y2 = y_center + box_height / 2

            ground_truth.append(
                {
                    "class_id": class_id,
                    "box": [x1, y1, x2, y2],
                }
            )

    return ground_truth


# ============================================================
# Match predictions with ground truth
# ============================================================

def match_predictions(predictions, ground_truth):
    """
    Match predictions with ground-truth boxes.

    A prediction is a true positive when:
      1. Predicted class is correct.
      2. IoU >= MATCH_IOU_THRESHOLD.
      3. Ground-truth box has not already been matched.
    """

    true_positives = 0
    false_positives = 0
    matched_ious = []

    matched_ground_truth = set()

    # Highest-confidence predictions first
    predictions = sorted(
        predictions,
        key=lambda item: item["confidence"],
        reverse=True,
    )

    for prediction in predictions:
        best_iou = 0.0
        best_ground_truth_index = -1

        for gt_index, gt in enumerate(ground_truth):

            if gt_index in matched_ground_truth:
                continue

            if prediction["class_id"] != gt["class_id"]:
                continue

            current_iou = calculate_iou(
                prediction["box"],
                gt["box"],
            )

            if current_iou > best_iou:
                best_iou = current_iou
                best_ground_truth_index = gt_index

        if (
            best_ground_truth_index >= 0
            and best_iou >= MATCH_IOU_THRESHOLD
        ):
            true_positives += 1
            matched_ground_truth.add(best_ground_truth_index)
            matched_ious.append(best_iou)

        else:
            false_positives += 1

    false_negatives = len(ground_truth) - len(matched_ground_truth)

    return (
        true_positives,
        false_positives,
        false_negatives,
        matched_ious,
    )


# ============================================================
# Standard Ultralytics metrics
# ============================================================

def calculate_standard_metrics(model):
    """
    Calculate standard YOLO metrics:
      Precision
      Recall
      mAP50
      mAP75
      mAP50-95
    """

    print("\n" + "=" * 70)
    print("STANDARD YOLO TEST EVALUATION")
    print("=" * 70)

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,

        # Keep low confidence for AP calculation
        conf=0.001,

        # NMS IoU threshold
        iou=NMS_IOU_THRESHOLD,

        max_det=MAX_DETECTIONS,

        plots=True,
        save_json=False,

        project=str(PROJECT_DIR),
        name=f"{EXPERIMENT_NAME}_standard_test",
        exist_ok=True,
        verbose=True,
    )

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)

    f1_score = (
        2 * precision * recall
        / (precision + recall + 1e-16)
    )

    results = {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "mAP50_95": float(metrics.box.map),
    }

    return results


# ============================================================
# Custom accuracy and IoU evaluation
# ============================================================

def calculate_custom_metrics(model):
    """
    Calculate custom metrics at:

        Confidence >= 0.25
        IoU >= 0.45

    Detection accuracy:

        Accuracy = TP / (TP + FP + FN)

    True negatives are not normally defined for object detection.
    """

    print("\n" + "=" * 70)
    print("CUSTOM TEST EVALUATION")
    print("=" * 70)

    image_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff",
    }

    image_paths = sorted(
        [
            path
            for path in TEST_IMAGES.rglob("*")
            if path.suffix.lower() in image_extensions
        ]
    )

    if not image_paths:
        raise FileNotFoundError(
            f"No test images were found in:\n{TEST_IMAGES}"
        )

    total_tp = 0
    total_fp = 0
    total_fn = 0

    all_matched_ious = []
    per_image_results = []

    prediction_results = model.predict(
        source=[str(path) for path in image_paths],
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        iou=NMS_IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        device=DEVICE,
        stream=True,

        save=True,
        save_txt=True,
        save_conf=True,

        project=str(PROJECT_DIR),
        name=f"{EXPERIMENT_NAME}_test_predictions",
        exist_ok=True,

        verbose=False,
    )

    for image_path, result in zip(
        image_paths,
        prediction_results,
    ):
        image = cv2.imread(str(image_path))

        if image is None:
            print(f"Could not read image: {image_path}")
            continue

        image_height, image_width = image.shape[:2]

        label_path = (
            TEST_LABELS
            / f"{image_path.stem}.txt"
        )

        ground_truth = load_ground_truth(
            label_path=label_path,
            image_width=image_width,
            image_height=image_height,
        )

        predictions = []

        if result.boxes is not None:
            for box in result.boxes:

                coordinates = (
                    box.xyxy[0]
                    .detach()
                    .cpu()
                    .numpy()
                    .tolist()
                )

                confidence = float(
                    box.conf[0]
                    .detach()
                    .cpu()
                    .item()
                )

                class_id = int(
                    box.cls[0]
                    .detach()
                    .cpu()
                    .item()
                )

                predictions.append(
                    {
                        "class_id": class_id,
                        "confidence": confidence,
                        "box": coordinates,
                    }
                )

        tp, fp, fn, matched_ious = match_predictions(
            predictions=predictions,
            ground_truth=ground_truth,
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

        all_matched_ious.extend(matched_ious)

        image_precision = tp / (tp + fp + 1e-16)
        image_recall = tp / (tp + fn + 1e-16)

        image_f1 = (
            2 * image_precision * image_recall
            / (image_precision + image_recall + 1e-16)
        )

        image_accuracy = tp / (
            tp + fp + fn + 1e-16
        )

        image_mean_iou = (
            float(np.mean(matched_ious))
            if matched_ious
            else 0.0
        )

        per_image_results.append(
            {
                "image": image_path.name,
                "ground_truth": len(ground_truth),
                "predictions": len(predictions),
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "precision": image_precision,
                "recall": image_recall,
                "f1_score": image_f1,
                "accuracy": image_accuracy,
                "mean_iou": image_mean_iou,
            }
        )

    precision = total_tp / (
        total_tp + total_fp + 1e-16
    )

    recall = total_tp / (
        total_tp + total_fn + 1e-16
    )

    f1_score = (
        2 * precision * recall
        / (precision + recall + 1e-16)
    )

    # Detection accuracy/Jaccard accuracy
    accuracy = total_tp / (
        total_tp + total_fp + total_fn + 1e-16
    )

    mean_iou = (
        float(np.mean(all_matched_ious))
        if all_matched_ious
        else 0.0
    )

    median_iou = (
        float(np.median(all_matched_ious))
        if all_matched_ious
        else 0.0
    )

    results = {
        "confidence_threshold": CONF_THRESHOLD,
        "iou_threshold": MATCH_IOU_THRESHOLD,
        "test_images": len(image_paths),
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "accuracy": accuracy,
        "mean_iou": mean_iou,
        "median_iou": median_iou,
    }

    return results, per_image_results


# ============================================================
# Save metrics
# ============================================================

def save_results(
    standard_metrics,
    custom_metrics,
    per_image_results,
):
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    overall_csv = OUTPUT_DIR / "overall_metrics.csv"

    combined_results = {
        "standard_precision": standard_metrics["precision"],
        "standard_recall": standard_metrics["recall"],
        "standard_f1_score": standard_metrics["f1_score"],
        "mAP50": standard_metrics["mAP50"],
        "mAP75": standard_metrics["mAP75"],
        "mAP50_95": standard_metrics["mAP50_95"],

        "confidence_threshold": custom_metrics[
            "confidence_threshold"
        ],
        "iou_threshold": custom_metrics["iou_threshold"],
        "TP": custom_metrics["TP"],
        "FP": custom_metrics["FP"],
        "FN": custom_metrics["FN"],
        "custom_precision": custom_metrics["precision"],
        "custom_recall": custom_metrics["recall"],
        "custom_f1_score": custom_metrics["f1_score"],
        "detection_accuracy": custom_metrics["accuracy"],
        "mean_matched_iou": custom_metrics["mean_iou"],
        "median_matched_iou": custom_metrics[
            "median_iou"
        ],
    }

    with overall_csv.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=combined_results.keys(),
        )

        writer.writeheader()
        writer.writerow(combined_results)

    per_image_csv = OUTPUT_DIR / "per_image_metrics.csv"

    if per_image_results:
        with per_image_csv.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=per_image_results[0].keys(),
            )

            writer.writeheader()
            writer.writerows(per_image_results)

    print(f"\nOverall metrics saved to:\n{overall_csv}")
    print(f"\nPer-image metrics saved to:\n{per_image_csv}")


# ============================================================
# Print results
# ============================================================

def print_results(
    standard_metrics,
    custom_metrics,
):
    print("\n" + "=" * 70)
    print("FINAL TEST RESULTS")
    print("=" * 70)

    print("\nSTANDARD YOLO METRICS")
    print("-" * 70)

    print(
        f"Precision:          "
        f"{standard_metrics['precision']:.6f}"
    )

    print(
        f"Recall:             "
        f"{standard_metrics['recall']:.6f}"
    )

    print(
        f"F1-score:           "
        f"{standard_metrics['f1_score']:.6f}"
    )

    print(
        f"mAP@0.50:           "
        f"{standard_metrics['mAP50']:.6f}"
    )

    print(
        f"mAP@0.75:           "
        f"{standard_metrics['mAP75']:.6f}"
    )

    print(
        f"mAP@0.50:0.95:      "
        f"{standard_metrics['mAP50_95']:.6f}"
    )

    print("\nCUSTOM METRICS AT IoU >= 0.45")
    print("-" * 70)

    print(f"TP:                 {custom_metrics['TP']}")
    print(f"FP:                 {custom_metrics['FP']}")
    print(f"FN:                 {custom_metrics['FN']}")

    print(
        f"Precision:          "
        f"{custom_metrics['precision']:.6f}"
    )

    print(
        f"Recall:             "
        f"{custom_metrics['recall']:.6f}"
    )

    print(
        f"F1-score:           "
        f"{custom_metrics['f1_score']:.6f}"
    )

    print(
        f"Detection Accuracy: "
        f"{custom_metrics['accuracy']:.6f}"
    )

    print(
        f"Mean matched IoU:   "
        f"{custom_metrics['mean_iou']:.6f}"
    )

    print(
        f"Median matched IoU: "
        f"{custom_metrics['median_iou']:.6f}"
    )

    print(
        f"Confidence threshold: "
        f"{custom_metrics['confidence_threshold']}"
    )

    print(
        f"IoU threshold:        "
        f"{custom_metrics['iou_threshold']}"
    )


# ============================================================
# Main
# ============================================================

def main():
    if not DATA_YAML.exists():
        raise FileNotFoundError(
            f"data.yaml was not found:\n{DATA_YAML}"
        )

    if not BEST_CHECKPOINT.exists():
        raise FileNotFoundError(
            f"best.pt was not found:\n{BEST_CHECKPOINT}"
        )

    if not TEST_IMAGES.exists():
        raise FileNotFoundError(
            f"Test image directory was not found:\n"
            f"{TEST_IMAGES}"
        )

    if not TEST_LABELS.exists():
        raise FileNotFoundError(
            f"Test label directory was not found:\n"
            f"{TEST_LABELS}"
        )

    print("=" * 70)
    print("LOADING BEST CHECKPOINT")
    print("=" * 70)
    print(BEST_CHECKPOINT)

    model = YOLO(str(BEST_CHECKPOINT))

    standard_metrics = calculate_standard_metrics(model)

    custom_metrics, per_image_results = (
        calculate_custom_metrics(model)
    )

    print_results(
        standard_metrics=standard_metrics,
        custom_metrics=custom_metrics,
    )

    save_results(
        standard_metrics=standard_metrics,
        custom_metrics=custom_metrics,
        per_image_results=per_image_results,
    )

    print("\n" + "=" * 70)
    print("TESTING COMPLETED")
    print("=" * 70)


if __name__ == "__main__":
    main()

LOADING BEST CHECKPOINT
D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Experiments\YOLOv9m_Axial_Localization\weights\best.pt

STANDARD YOLO TEST EVALUATION
Ultralytics 8.3.179  Python-3.12.7 torch-2.11.0.dev20260128+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
YOLOv9s summary (fused): 197 layers, 7,169,023 parameters, 0 gradients, 26.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 974.5404.1 MB/s, size: 80.0 KB)


val: Scanning D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Axial_Localization\labels\test.cache... 499 images, 0 backgrounds, 0 corrupt: 100%|██████████| 499/499 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [00:02<00:00, 13.58it/s]


                   all        499       2491      0.971      0.969      0.976      0.529
                   CCS        499        499      0.998      0.987      0.995       0.62
                   LLS        498        498       0.99      0.983      0.993      0.556
                   RLS        498        498       0.97      0.969      0.979      0.546
                   RFS        498        498      0.927      0.934      0.937      0.454
                   LFS        498        498      0.968      0.972      0.975      0.467
Speed: 0.0ms preprocess, 1.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\YOLO_Experiments\YOLOv9m_Axial_Localization_standard_test

CUSTOM TEST EVALUATION

FINAL TEST RESULTS

STANDARD YOLO METRICS
----------------------------------------------------------------------
Precision:          0.970541
Recall:             0.968844
F1-score:           0.9696